# 02 — Event memory: store → retrieve → inspect

Day 1–3 cells use SQLite and keyword retrieval. Day 4 cells add semantic retrieval with MiniLM CPU embeddings (`sentence-transformers/all-MiniLM-L6-v2`). No Groq key is needed. Locally, install MiniLM with `pip install -r backend/requirements-ml.txt`; without it the semantic cells skip and retrieval uses the keyword fallback.

In [ ]:
import sys, subprocess, tempfile
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend/app/world.py").exists()), None)
if ROOT is None and "google.colab" in sys.modules:
    ROOT = Path("/content/MiniDavis")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "https://github.com/KTH-Sys/MiniDavis.git", str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "backend/requirements-ml.txt")], check=True)
if ROOT is None:
    raise RuntimeError("Open this notebook from a MiniDavis checkout.")
sys.path.insert(0, str(ROOT))
print("Project imports ready.")

In [ ]:
from backend.app.memory import EventStore
from backend.app.schemas import Event, StepRequest
from backend.app.world import World
path = Path(tempfile.mkdtemp()) / "memory.sqlite"
store = EventStore(path)
event = Event(event_id="sample:1:ethan", run_id="sample", tick=1, agent_id="ethan", action="gather", outcome="accepted", text="Ethan gathered two wood.")
store.save(event)
store.close()
store = EventStore(path)
assert store.get(event.event_id) == event
print("Stored and reloaded:", event.event_id)


In [ ]:
rejected = Event(event_id="sample:2:ethan", run_id="sample", tick=2, agent_id="ethan", action="move", outcome="rejected", text="Ethan could not move outside the map.")
store.save(rejected)
assert store.get(rejected.event_id).outcome == "rejected"
assert store.get(event.event_id).outcome == "accepted"
context = store.retrieve("sample", "ethan", "wood", as_of_tick=2)
assert event.event_id in [e.event_id for e in context.relevant]
assert not store.retrieve("sample", "gayatri", "wood", 2).recent
print("Accepted/rejected outcomes and private memory scope verified.")


In [ ]:
world = World(seed=11, store=store)
for index in range(2):
    state = world.snapshot()
    state = world.step(StepRequest(run_id=state.run_id, expected_revision=state.revision, request_id=f"memory-{index}"))
for resident in state.agents:
    decision = world.inspect(resident.id).decision
    assert decision.memory.recent
    assert all(store.get(e.event_id) == e for e in decision.memory.recent)
    print(resident.name, [e.event_id for e in decision.memory.recent])
store.close()


## Day 4 — semantic retrieval with MiniLM

Each event is turned into an embedding: a list of 384 numbers where similar meanings get similar numbers. The query is embedded too, and events are ranked by cosine similarity. Run, resident and tick are still filtered in SQL **before** ranking, so another resident's private memory is never ranked. Event embeddings are cached in SQLite, so each event is embedded once. If the model fails, retrieval falls back to keyword search.

In [ ]:
import json
from backend.app.memory import minilm_embedder
embedder = minilm_embedder()
if embedder is None:
    print("MiniLM not installed: semantic cells skipped, keyword fallback in use.")
else:
    store = EventStore(Path(tempfile.mkdtemp()) / "semantic.sqlite", embedder=embedder)
    store.save(Event(event_id="s:1:ethan", run_id="s", tick=1, agent_id="ethan", action="gather", outcome="accepted", text="Ethan gathered two logs for shelter."))
    store.save(Event(event_id="s:2:ethan", run_id="s", tick=2, agent_id="ethan", action="eat", outcome="accepted", text="Ethan ate lunch and recovered energy."))
    store.save(Event(event_id="s:1:gayatri", run_id="s", tick=1, agent_id="gayatri", action="gather", outcome="accepted", text="Gayatri gathered timber."))
    context = store.retrieve("s", "ethan", "timber", as_of_tick=2)
    assert context.mode == "semantic"
    assert context.relevant[0].event_id == "s:1:ethan"  # no shared word with "timber"
    assert all(e.agent_id == "ethan" for e in context.relevant)  # Gayatri's private event is excluded
    keyword = EventStore(Path(tempfile.mkdtemp()) / "keyword.sqlite")
    keyword.save_many([store.get("s:1:ethan"), store.get("s:2:ethan")])
    print("Keyword search for 'timber':", [e.event_id for e in keyword.retrieve("s", "ethan", "timber", 2).relevant])
    print("MiniLM search for 'timber': ", [e.event_id for e in context.relevant])
    keyword.close(); store.close()

In [ ]:
# The 10 frozen retrieval checks: pass = at least 8 expected events in the top three.
if embedder is not None:
    fixtures = json.loads((ROOT / "ml/data/retrieval_examples.json").read_text())
    store = EventStore(Path(tempfile.mkdtemp()) / "frozen.sqlite", embedder=embedder)
    for tick, row in enumerate(fixtures, 1):
        store.save(Event(event_id=f"fixture:{tick}:ethan", run_id="fixture", agent_id="ethan", tick=tick,
                         action=row["action"], outcome=row["outcome"], text=row["text"]))
    hits = sum(f"fixture:{tick}:ethan" in [e.event_id for e in store.retrieve("fixture", "ethan", row["query"], 10).relevant]
               for tick, row in enumerate(fixtures, 1))
    store.close()
    print(f"Frozen retrieval checks: {hits}/10 expected events in the top three")
    assert hits >= 8

These facts are supplied to the agent input. The baseline receives them but uses fixed rules. The Groq adapter uses the bounded subset shown in the inspector; retrieval alone is not proof that memory changed a decision.